# Depuración de duplicados de BRISC

**Código auxiliar de depuración de duplicados**

Este notebook identifica imágenes duplicadas en las particiones de entrenamiento y prueba de la tarea de segmentación de BRISC, conservando la correspondencia entre imágenes y máscaras. La detección combina coincidencias exactas mediante SHA-256 y comparación de píxeles, y candidatos perceptuales mediante pHash y SSIM.

La depuración aplica tres reglas jerárquicas: exclusión de grupos con etiquetas inconsistentes, exclusión cuando la concordancia entre máscaras es insuficiente (Dice inferior a 0,80) y conservación de una única copia en los demás casos, priorizando la partición de prueba. Se registran las decisiones, el balance antes y después del procedimiento y las galerías de los grupos identificados. El conjunto original permanece inalterado.

Para reproducir la depuración histórica se requiere una copia de BRISC anterior a la eliminación de duplicados, situada en `data/BRISC2025_original/`. La copia depurada ubicada en `data/BRISC2025/` no se utiliza como entrada ni se modifica.


## 0. Bibliotecas y configuración

Se establecen las rutas del repositorio, los criterios de detección de duplicados y las opciones de exportación. La creación de una nueva copia depurada es opcional y no sobrescribe datos existentes.


In [ ]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations, product
import hashlib
import json
import math
import os
import re
import shutil

import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn
from skimage.metrics import structural_similarity
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)
Image.MAX_IMAGE_PIXELS = None


In [ ]:
def localizar_raiz_repositorio():
    """Localiza la raíz del repositorio desde JupyterLab o el directorio del notebook."""
    directorio_actual = Path.cwd().resolve()
    for candidato in (directorio_actual, *directorio_actual.parents):
        if (candidato / "notebooks" / "OE0").is_dir():
            return candidato
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. "
        "Se requiere la carpeta notebooks/OE0; ejecutar desde la raíz "
        "del repositorio o desde alguno de sus subdirectorios."
    )


REPO_ROOT = localizar_raiz_repositorio()
# Datos originales, copia depurada y resultados, relativos al repositorio.
ORIGEN_ROOT = REPO_ROOT / "data" / "BRISC2025_original"
DESTINO_ROOT = REPO_ROOT / "data" / "BRISC2025"
ORIGEN = ORIGEN_ROOT / "segmentation_task"
DESTINO = DESTINO_ROOT / "segmentation_task"
RESULTADOS_OE0 = REPO_ROOT / "results" / "OE0"
SALIDA = RESULTADOS_OE0 / "depuracion_duplicados"

# Máscaras y consistencia de anotaciones.
UMBRAL_MASCARA = 127
UMBRAL_DICE_MINIMO = 0.80
TAM_DICE_PERCEPTUAL = (256, 256)  # ancho, alto; vecino más cercano

# Parámetros usados en la búsqueda de candidatos visualmente confirmados.
DISTANCIA_PHASH_MAX = 6
SSIM_MINIMO = 0.90
DIFERENCIA_MAX_ASPECTO = 0.02
TAM_SSIM = (128, 128)
PARES_POR_PAGINA = 5
DPI_GALERIA = 115

# Aplicar las reglas a los candidatos perceptuales requiere verificación visual
# de los pares detectados utilizando esta configuración y esta base de datos.
APLICAR_PERCEPTUALES_CONFIRMADOS = False

# Permite comprobar que la lista de candidatos coincide con la revisión visual.
N_PARES_PERCEPTUALES_ESPERADOS = None

# Por defecto se generan resultados sin reconstruir ni sobrescribir el conjunto depurado.
# Para crear una copia nueva, establecer True si el destino todavía no existe.
CREAR_DATASET_DEPURADO = False

if not ORIGEN.is_dir():
    raise FileNotFoundError(
        f"No existe el conjunto original: {ORIGEN}. "
        "Ubicar los datos BRISC originales en data/BRISC2025_original/segmentation_task."
    )
if ORIGEN_ROOT.resolve() == DESTINO_ROOT.resolve():
    raise ValueError("El origen no puede coincidir con el destino")
if (ORIGEN_ROOT.resolve() in DESTINO_ROOT.resolve().parents
        or DESTINO_ROOT.resolve() in ORIGEN_ROOT.resolve().parents):
    raise ValueError("Origen y destino no pueden contenerse mutuamente")
if not (0 <= UMBRAL_MASCARA <= 255 and 0 <= UMBRAL_DICE_MINIMO <= 1):
    raise ValueError("Umbrales de máscara/Dice fuera de rango")
if not (0 <= DISTANCIA_PHASH_MAX <= 63 and -1 <= SSIM_MINIMO <= 1):
    raise ValueError("pHash (63 bits) o SSIM fuera de rango")
if PARES_POR_PAGINA <= 0:
    raise ValueError("PARES_POR_PAGINA debe ser positivo")

SALIDA.mkdir(parents=True, exist_ok=True)
DIR_EXACTOS = SALIDA / "01_duplicados_exactos"
DIR_PERCEPTUALES = SALIDA / "02_duplicados_perceptuales"
DIR_BALANCE = SALIDA / "03_tablas_balance"
DIR_REGISTROS = SALIDA / "04_registros_depuracion"
SUBCARPETAS_REGLAS = {
    "R1": "regla_1_etiquetas_inconsistentes",
    "R2": "regla_2_mascaras_inconsistentes",
    "R3": "regla_3_duplicados_redundantes",
}
for directorio in (DIR_EXACTOS, DIR_PERCEPTUALES):
    for subcarpeta in SUBCARPETAS_REGLAS.values():
        (directorio / subcarpeta).mkdir(parents=True, exist_ok=True)
for directorio in (DIR_BALANCE, DIR_REGISTROS):
    directorio.mkdir(parents=True, exist_ok=True)
print("Origen:", ORIGEN)
print("Destino:", DESTINO)
print("Resultados:", SALIDA)
print("Aplicar reglas a perceptuales:", APLICAR_PERCEPTUALES_CONFIRMADOS)
print("Crear dataset depurado:", CREAR_DATASET_DEPURADO)


## 1. Inventario de imágenes y máscaras

Se enumeran los pares de `segmentation_task/train` y `segmentation_task/test` y se validan sus nombres, dimensiones y correspondencia. Se calculan huellas SHA-256 del archivo y de los píxeles decodificados.


In [ ]:
EXTENSIONES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
CODIGO_TUMOR = {"gl": "glioma", "me": "meningioma", "pi": "pituitario"}
CODIGO_PLANO = {"ax": "axial", "co": "coronal", "sa": "sagital"}
PATRON = re.compile(
    r"^brisc2025_(train|test)_\d+_(gl|me|pi)_(ax|co|sa)_t1$", re.IGNORECASE
)
PARTICIONES = ("train", "test")


def sha_archivo(ruta, tam=1024 * 1024):
    h = hashlib.sha256()
    with ruta.open("rb") as f:
        for bloque in iter(lambda: f.read(tam), b""):
            h.update(bloque)
    return h.hexdigest()


def sha_matriz(matriz, etiqueta):
    """SHA-256 con forma y tipo de dato para evitar ambigüedades."""
    a = np.ascontiguousarray(matriz)
    h = hashlib.sha256()
    h.update(f"{etiqueta}|{a.shape}|{a.dtype.str}|".encode("ascii"))
    h.update(a.tobytes())
    return h.hexdigest()


def buscar_mascara(ruta_imagen, split):
    directorio = ORIGEN / split / "masks"
    candidatos = sorted(p for p in directorio.glob(ruta_imagen.stem + ".*")
                        if p.is_file() and p.suffix.lower() in EXTENSIONES)
    if len(candidatos) != 1:
        raise ValueError(f"Se esperaban exactamente 1 máscara para {ruta_imagen.name}, se encontraron {len(candidatos)}")
    return candidatos[0]


def procesar_par(ruta_img, split):
    coincidencia = PATRON.fullmatch(ruta_img.stem)
    if coincidencia is None:
        raise ValueError(f"Nombre no reconocido: {ruta_img.name}")
    if coincidencia.group(1).lower() != split:
        raise ValueError(f"Nombre/partición discordantes: {ruta_img.name} / {split}")
    ruta_mask = buscar_mascara(ruta_img, split)
    with Image.open(ruta_img) as obj:
        rgb = np.asarray(obj.convert("RGB"), dtype=np.uint8)
    with Image.open(ruta_mask) as obj:
        mask = np.asarray(obj.convert("L"), dtype=np.uint8)
    if mask.shape != rgb.shape[:2]:
        raise ValueError(f"Imagen y máscara con dimensiones diferentes: {ruta_img} / {ruta_mask}")
    mask_binaria = (mask > UMBRAL_MASCARA).astype(np.uint8)
    relative_img = ruta_img.relative_to(ORIGEN)
    relative_mask = ruta_mask.relative_to(ORIGEN)
    return {
        "ruta_relativa_imagen": relative_img.as_posix(),
        "ruta_relativa_mascara": relative_mask.as_posix(),
        "ruta_imagen_original": str(ruta_img),
        "ruta_mascara_original": str(ruta_mask),
        "particion": split,
        "tipo_tumor": CODIGO_TUMOR[coincidencia.group(2).lower()],
        "plano": CODIGO_PLANO[coincidencia.group(3).lower()],
        "sha256_archivo": sha_archivo(ruta_img),
        "sha256_pixeles": sha_matriz(rgb, "RGB"),
        "sha256_mascara_binaria": sha_matriz(mask_binaria, "MASCARA_BINARIA"),
        "alto": int(rgb.shape[0]), "ancho": int(rgb.shape[1]),
        "area_tumoral_pixeles": int(mask_binaria.sum()),
    }


rutas_imagenes = []
for split in PARTICIONES:
    carpeta_imagenes = ORIGEN / split / "images"
    carpeta_mascaras = ORIGEN / split / "masks"
    if not carpeta_imagenes.is_dir() or not carpeta_mascaras.is_dir():
        raise FileNotFoundError(f"Carpetas imágenes/máscaras ausentes en {split}")
    rutas_imagenes.extend((p, split) for p in sorted(carpeta_imagenes.iterdir())
                          if p.is_file() and p.suffix.lower() in EXTENSIONES)

registros = []
problemas = []
for numero, (ruta_imagen, split) in enumerate(rutas_imagenes, 1):
    try:
        registros.append(procesar_par(ruta_imagen, split))
    except Exception as exc:
        problemas.append({"ruta_imagen": str(ruta_imagen), "problema": f"{type(exc).__name__}: {exc}"})
    if numero % 500 == 0 or numero == len(rutas_imagenes):
        print(f"Pares inspeccionados: {numero:,}/{len(rutas_imagenes):,}")

pd.DataFrame(problemas, columns=["ruta_imagen", "problema"]).to_csv(
    DIR_REGISTROS / "problemas_prevalidacion.csv", index=False, encoding="utf-8-sig"
)
if problemas:
    raise RuntimeError(
        f"Se encontraron {len(problemas)} problemas. No crear la base depurada hasta resolverlos; ver 04_registros_depuracion/problemas_prevalidacion.csv"
    )
if not registros:
    raise RuntimeError("No se encontraron imágenes para depurar")

inventario = pd.DataFrame(registros).sort_values("ruta_relativa_imagen").reset_index(drop=True)
inventario.insert(0, "id", np.arange(len(inventario), dtype=int))
inventario.to_csv(DIR_REGISTROS / "inventario_original.csv", index=False, encoding="utf-8-sig")
print(f"Pares válidos: {len(inventario):,}")
display(inventario.groupby(["particion", "tipo_tumor"]).size().unstack(fill_value=0))

## 2. Identificación de duplicados exactos

Se agrupan las imágenes con archivos idénticos o matrices de píxeles iguales mediante una estructura de conjuntos disjuntos.


In [ ]:
# Estructura de conjuntos disjuntos (unión-búsqueda) para agrupar copias exactas.
n = len(inventario)
padre = list(range(n))
tamano = [1] * n

def raiz(i):
    while padre[i] != i:
        padre[i] = padre[padre[i]]
        i = padre[i]
    return i

def unir(a, b):
    ra, rb = raiz(a), raiz(b)
    if ra == rb:
        return
    if tamano[ra] < tamano[rb]:
        ra, rb = rb, ra
    padre[rb] = ra
    tamano[ra] += tamano[rb]

for campo in ("sha256_archivo", "sha256_pixeles"):
    for _, valores in inventario.groupby(campo):
        ids = valores["id"].tolist()
        for j in ids[1:]:
            unir(ids[0], j)

componentes = defaultdict(list)
for i in range(n):
    componentes[raiz(i)].append(i)

componentes_duplicados = sorted((members for members in componentes.values() if len(members) > 1),
                               key=lambda ids: inventario.at[min(ids), "ruta_relativa_imagen"])
grupo_por_id = {}
for k, ids in enumerate(componentes_duplicados, 1):
    for i in ids:
        grupo_por_id[i] = f"G{k:05d}"

print(f"Grupos exactos: {len(componentes_duplicados)}")
print(f"Imágenes en grupos exactos: {sum(map(len, componentes_duplicados))}")
print(f"Imágenes sin duplicados exactos: {n-sum(map(len, componentes_duplicados))}")

## 3. Depuración de duplicados exactos

Se aplican las reglas de consistencia de etiquetas (R1), concordancia de máscaras mediante Dice (R2) y conservación de una copia representativa (R3).


In [ ]:
particion_prioridad = {"test": 0, "train": 1}

# Precarga únicamente de máscaras pertenecientes a grupos repetidos.
mascaras_binarias = {}

def cargar_mascara_binaria(i):
    if i not in mascaras_binarias:
        with Image.open(inventario.at[i, "ruta_mascara_original"]) as imagen:
            mascara = np.asarray(imagen.convert("L")) > UMBRAL_MASCARA
        mascaras_binarias[i] = mascara
    return mascaras_binarias[i]


def dice_binario(a, b):
    if a.shape != b.shape:
        raise ValueError("Máscaras de distinto tamaño en un grupo de imágenes idénticas")
    na, nb = int(np.count_nonzero(a)), int(np.count_nonzero(b))
    if na + nb == 0:
        return 1.0
    return 2.0 * int(np.count_nonzero(a & b)) / (na + nb)


def clave_prioridad(i):
    return (
        particion_prioridad[inventario.at[i, "particion"]],
        -int(inventario.at[i, "area_tumoral_pixeles"]),
        inventario.at[i, "ruta_relativa_imagen"].lower(),
    )


decision = {
    i: {"grupo_exacto": grupo_por_id.get(i, ""),
        "decision": "conservar", "motivo": "sin_duplicado_exacto"}
    for i in range(n)
}
resumen_grupos = []
comparaciones_dice = []

for ids in componentes_duplicados:
    filas = inventario.loc[ids]
    grupo = grupo_por_id[ids[0]]

    # REGLA 1. Si hay discordancia, no se comparan máscaras ni se elige copia.
    dif_etiquetas = filas[["tipo_tumor", "plano"]].drop_duplicates().shape[0] > 1
    dice_minimo = np.nan
    par_peor = (None, None)
    if dif_etiquetas:
        regla = "R1_etiquetas_inconsistentes"
        elegido = None
    else:
        # REGLA 2. Comparación exhaustiva por parejas, NO mediante igualdad de hashes.
        for i, j in combinations(ids, 2):
            dice = dice_binario(cargar_mascara_binaria(i), cargar_mascara_binaria(j))
            comparaciones_dice.append({
                "grupo_exacto": grupo,
                "id_1": i, "id_2": j,
                "ruta_1": inventario.at[i, "ruta_relativa_imagen"],
                "ruta_2": inventario.at[j, "ruta_relativa_imagen"],
                "particion_1": inventario.at[i, "particion"],
                "particion_2": inventario.at[j, "particion"],
                "area_tumoral_1": int(inventario.at[i, "area_tumoral_pixeles"]),
                "area_tumoral_2": int(inventario.at[j, "area_tumoral_pixeles"]),
                "dice": float(dice),
            })
            if np.isnan(dice_minimo) or dice < dice_minimo:
                dice_minimo = float(dice)
                par_peor = (i, j)

        if dice_minimo < UMBRAL_DICE_MINIMO:
            regla = "R2_dice_inferior_umbral"
            elegido = None
        else:
            # REGLA 3. Siempre prevalece test. El tamaño desempata solo dentro del set.
            regla = "R3_copias_aceptadas"
            elegido = min(ids, key=clave_prioridad)

    for i in ids:
        if elegido is None:
            decision[i].update(decision="excluir", motivo=regla)
        elif i == elegido:
            decision[i].update(decision="conservar", motivo="R3_copia_representativa")
        else:
            decision[i].update(decision="excluir", motivo="R3_copia_redundante")

    resumen_grupos.append({
        "grupo_exacto": grupo,
        "n_miembros": len(ids),
        "discordancia_etiquetas": bool(dif_etiquetas),
        "dice_minimo": dice_minimo,
        "umbral_dice": UMBRAL_DICE_MINIMO,
        "peor_par_id_1": par_peor[0],
        "peor_par_id_2": par_peor[1],
        "regla_aplicada": regla,
        "ruta_conservada": inventario.at[elegido, "ruta_relativa_imagen"] if elegido is not None else "",
        "area_tumoral_conservada": int(inventario.at[elegido, "area_tumoral_pixeles"]) if elegido is not None else np.nan,
        "n_eliminadas": len(ids) if elegido is None else len(ids) - 1,
    })

registro_decisiones = inventario.copy()
registro_decisiones["grupo_exacto"] = [decision[i]["grupo_exacto"] for i in range(n)]
registro_decisiones["decision"] = [decision[i]["decision"] for i in range(n)]
registro_decisiones["motivo"] = [decision[i]["motivo"] for i in range(n)]
registro_decisiones["ruta_nueva_imagen"] = registro_decisiones["ruta_relativa_imagen"].map(
    lambda p: str(DESTINO / Path(p))
)
registro_decisiones["ruta_nueva_mascara"] = registro_decisiones["ruta_relativa_mascara"].map(
    lambda p: str(DESTINO / Path(p))
)

COLUMNAS_GRUPOS = [
    "grupo_exacto", "n_miembros", "discordancia_etiquetas", "dice_minimo", "umbral_dice",
    "peor_par_id_1", "peor_par_id_2", "regla_aplicada", "ruta_conservada",
    "area_tumoral_conservada", "n_eliminadas",
]
COLUMNAS_DICE = [
    "grupo_exacto", "id_1", "id_2", "ruta_1", "ruta_2", "particion_1", "particion_2",
    "area_tumoral_1", "area_tumoral_2", "dice",
]
grupos = pd.DataFrame(resumen_grupos, columns=COLUMNAS_GRUPOS)
pares_dice = pd.DataFrame(comparaciones_dice, columns=COLUMNAS_DICE)
conservadas = registro_decisiones.loc[registro_decisiones.decision == "conservar"].copy()
excluidas = registro_decisiones.loc[registro_decisiones.decision == "excluir"].copy()

for nombre, tabla in [
    ("etapa_exactos_decisiones_por_imagen.csv", registro_decisiones),
    ("grupos_exactos_y_reglas.csv", grupos),
    ("comparaciones_dice_pares.csv", pares_dice),
    ("etapa_exactos_imagenes_conservadas.csv", conservadas),
    ("etapa_exactos_imagenes_excluidas.csv", excluidas),
]:
    tabla.to_csv(DIR_REGISTROS / nombre, index=False, encoding="utf-8-sig")
resumen_motivos = registro_decisiones.groupby(["decision", "motivo"]).size().rename("n_imagenes").reset_index()
resumen_motivos.to_csv(DIR_REGISTROS / "etapa_exactos_resumen_decisiones.csv", index=False, encoding="utf-8-sig")

# Verificaciones invariantes de las reglas jerárquicas.
assert len(conservadas) + len(excluidas) == len(inventario)
assert not conservadas.duplicated("sha256_pixeles").any()
assert not conservadas.duplicated("sha256_archivo").any()
assert set(conservadas.id).isdisjoint(set(excluidas.id))
for ids in componentes_duplicados:
    t = registro_decisiones.loc[ids]
    assert (t.decision == "conservar").sum() <= 1

print(f"Imágenes originales: {len(inventario):,}")
print(f"Excluidas: {len(excluidas):,} | Conservadas: {len(conservadas):,}")
display(resumen_motivos)
if not grupos.empty:
    display(grupos.groupby("regla_aplicada").agg(
        n_grupos=("grupo_exacto", "size"), n_eliminadas=("n_eliminadas", "sum")
    ).reset_index())
    display(grupos[["grupo_exacto", "n_miembros", "dice_minimo", "regla_aplicada", "ruta_conservada"]].head(25))

## 4. Galerías de duplicados exactos

Se generan figuras de los grupos identificados, señalando las imágenes conservadas y excluidas y, cuando corresponde, la comparación de máscaras.


In [ ]:
def imagen_con_contorno(ruta_imagen, ruta_mascara):
    with Image.open(ruta_imagen) as obj:
        imagen = np.asarray(obj.convert("L"))
    with Image.open(ruta_mascara) as obj:
        mascara = np.asarray(obj.convert("L")) > UMBRAL_MASCARA
    if imagen.shape != mascara.shape:
        raise ValueError("Dimensiones diferentes entre imagen y máscara")
    return imagen, mascara


from matplotlib.patches import Patch
from IPython.display import Image as NotebookImage


def mascara_para_comparacion(idx, tipo, usar_cuadricula):
    """La comparación R2 utiliza la misma representación que se usó para la decisión."""
    if tipo == "perceptual" and usar_cuadricula:
        return mascara_normalizada(idx)
    return cargar_mascara_binaria(idx)


def crear_galeria_grupo(ids, tipo, grupo, regla, elegido=None,
                        dice_minimo=np.nan, peor_par=(None, None),
                        diferentes_dimensiones=False, metricas_perceptuales=None):
    """Guarda una galería con la decisión y, en perceptuales, pHash/SSIM por par."""
    ids = sorted(ids, key=lambda i: clave_prioridad(i))
    posiciones = {idx: k for k, idx in enumerate(ids, 1)}
    n_cols = len(ids) + (1 if regla == "R2" else 0)

    # Mostrar los valores medidos de TODOS los pares perceptuales detectados
    # dentro del grupo. No inventar medidas de pares unidos por transitividad.
    metricas = metricas_perceptuales or []
    if tipo == "perceptual" and metricas:
        altura_pie = max(1.35, 0.65 + 0.35 * len(metricas) + (0.60 if regla == "R2" else 0))
        fig = plt.figure(figsize=(4.4*n_cols, 5.2 + altura_pie))
        grilla = fig.add_gridspec(2, n_cols, height_ratios=[5.2, altura_pie], hspace=0.4)
        axes = [fig.add_subplot(grilla[0, c]) for c in range(n_cols)]
        eje_metricas = fig.add_subplot(grilla[1, :])
        eje_metricas.axis("off")
        lineas = [
            f"Imágenes {posiciones[a]}–{posiciones[b]}: SSIM = {ssim:.4f}  |  distancia pHash = {distancia}"
            for a, b, ssim, distancia in metricas
        ]
        eje_metricas.text(
            0.5, 0.80, "Pares perceptuales detectados\n" + "\n".join(lineas),
            ha="center", va="top", fontsize=9, family="monospace",
            transform=eje_metricas.transAxes,
        )
    else:
        fig, ejes = plt.subplots(1, n_cols, figsize=(4.4*n_cols, 5.2), squeeze=False)
        axes = list(ejes[0])
        eje_metricas = None

    for ax, idx in zip(axes[:len(ids)], ids):
        f = inventario.loc[idx]
        imagen, mascara = imagen_con_contorno(
            Path(f.ruta_imagen_original), Path(f.ruta_mascara_original)
        )
        ax.imshow(imagen, cmap="gray", vmin=0, vmax=255)
        if mascara.any() and not mascara.all():
            ax.contour(mascara, levels=[0.5], colors=["#ff9d2e"], linewidths=0.9)
        retenida = (idx == elegido and regla == "R3")
        estado = "CONSERVADA" if retenida else "ELIMINADA"
        color = "#188268" if retenida else "#bb3b3b"
        etiqueta_par = f"Imagen {posiciones[idx]} | " if eje_metricas is not None else ""
        titulo = (
            f"{etiqueta_par}{estado} | {f.particion}\n"
            f"{Path(f.ruta_imagen_original).name}\n"
            f"{f.tipo_tumor} · {f.plano} · área={int(f.area_tumoral_pixeles)} px"
        )
        ax.set_title(titulo, fontsize=8, color=color, fontweight="bold")
        ax.axis("off")

    if regla == "R2":
        a, b = peor_par
        if a is None or b is None:
            raise AssertionError(f"R2 sin peor pareja para {grupo}")
        ma = mascara_para_comparacion(a, tipo, diferentes_dimensiones)
        mb = mascara_para_comparacion(b, tipo, diferentes_dimensiones)
        if ma.shape != mb.shape:
            raise AssertionError("R2 requiere máscaras en la misma cuadrícula")
        superposicion = np.zeros((*ma.shape, 3), dtype=float)
        superposicion[ma & mb] = [0.25, 0.8, 0.28]
        superposicion[ma & ~mb] = [1.0, 0.20, 0.20]
        superposicion[~ma & mb] = [0.25, 0.48, 1.0]
        axes[-1].imshow(superposicion)
        axes[-1].set_title(f"Peor par: {int(a)} / {int(b)}\nDice = {dice_minimo:.4f}", fontsize=10)
        axes[-1].axis("off")
        leyenda = [
            Patch(facecolor="#40cc47", label="Intersección"),
            Patch(facecolor="#ff3333", label=f"Solo máscara {int(a)}"),
            Patch(facecolor="#407aff", label=f"Solo máscara {int(b)}"),
        ]
        if eje_metricas is not None:
            eje_metricas.legend(handles=leyenda, loc="lower center", ncol=3,
                                fontsize=9, bbox_to_anchor=(0.5, -0.10))
        else:
            fig.legend(handles=leyenda, loc="lower center", ncol=3, fontsize=9)

    # Explicitar por qué se conservó la imagen representativa en R3.
    if regla == "R3":
        particiones = {inventario.at[i, "particion"] for i in ids}
        prioridad = "test" if "test" in particiones else "train"
        candidatos_set = [i for i in ids if inventario.at[i, "particion"] == prioridad]
        if len(candidatos_set) == 1:
            razon_r3 = "Prioridad a test" if len(particiones) > 1 else "Copia única en el set"
        else:
            areas = [int(inventario.at[i, "area_tumoral_pixeles"]) for i in candidatos_set]
            if tipo == "perceptual" and diferentes_dimensiones:
                areas = [int(mascara_normalizada(i).sum()) for i in candidatos_set]
            razon_r3 = ("Prioridad test; " if len(particiones) > 1 else "") + (
                "mayor área tumoral" if max(areas) != min(areas) else "desempate alfabético"
            )
    else:
        razon_r3 = ""
    explicaciones = {
        "R1": "Etiquetas discordantes — se excluye el grupo completo",
        "R2": f"Máscaras discordantes (Dice mínimo = {dice_minimo:.4f} < {UMBRAL_DICE_MINIMO:.2f}) — se excluye el grupo completo",
        "R3": f"Conservar una copia: {razon_r3}; eliminar las demás",
    }
    fig.suptitle(f"{grupo} | {tipo.upper()} | {regla}\n{explicaciones[regla]}", fontsize=12)
    if eje_metricas is not None:
        fig.tight_layout(rect=(0, 0, 1, 0.88), h_pad=2.5)
    else:
        fig.tight_layout(rect=(0, 0.10 if regla == "R2" else 0, 1, 0.88))
    directorio = (DIR_EXACTOS if tipo == "exacto" else DIR_PERCEPTUALES) / SUBCARPETAS_REGLAS[regla]
    ruta = directorio / f"grupo_{grupo}.png"
    fig.savefig(ruta, dpi=DPI_GALERIA, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return ruta


def preparar_directorios_galerias(directorio):
    """Evita mezclar figuras de dos ejecuciones sin tocar otras carpetas."""
    for subcarpeta in SUBCARPETAS_REGLAS.values():
        for figura in (directorio / subcarpeta).glob("grupo_*.png"):
            figura.unlink()


def mostrar_ejemplo_por_regla(ejemplos, titulo):
    print(titulo)
    for regla in ("R1", "R2", "R3"):
        rutas = ejemplos[regla]
        print(f"{regla}: {len(rutas)} galerías guardadas")
        if rutas:
            display(NotebookImage(filename=str(rutas[0]), width=950))


In [ ]:
preparar_directorios_galerias(DIR_EXACTOS)
indices_por_grupo_exacto = {grupo_por_id[ids[0]]: ids for ids in componentes_duplicados}
ejemplos_exactos = {r: [] for r in SUBCARPETAS_REGLAS}
for fila in grupos.itertuples(index=False):
    regla = fila.regla_aplicada[:2]
    ids = indices_por_grupo_exacto[fila.grupo_exacto]
    elegido = next((i for i in ids if decision[i]["decision"] == "conservar"), None)
    figura = crear_galeria_grupo(
        ids, "exacto", fila.grupo_exacto, regla,
        elegido=elegido, dice_minimo=fila.dice_minimo,
        peor_par=(fila.peor_par_id_1, fila.peor_par_id_2),
    )
    ejemplos_exactos[regla].append(figura)
mostrar_ejemplo_por_regla(ejemplos_exactos, "Galerías: duplicados exactos")


## 5. Detección de duplicados perceptuales

Se buscan candidatos entre las imágenes conservadas mediante pHash, SSIM y compatibilidad de la relación de aspecto.


In [ ]:
def phash_desde_gris_32(gris_32):
    """pHash basado en la DCT, con 63 coeficientes distintos de la componente DC."""
    bajos = dctn(gris_32.astype(np.float32), norm="ortho")[:8, :8].ravel()[1:]
    referencia = np.median(bajos)
    huella = 0
    for bit in bajos > referencia:
        huella = (huella << 1) | int(bit)
    return huella


def miniaturas_y_huella(ruta):
    with Image.open(ruta) as original:
        rgb = original.convert("RGB")
        ancho, alto = rgb.size
        gris = rgb.convert("L")
        reducido = np.asarray(gris.resize((32, 32), Image.Resampling.LANCZOS), dtype=np.uint8)
        miniatura = np.asarray(gris.resize(TAM_SSIM, Image.Resampling.BILINEAR), dtype=np.uint8)
    return phash_desde_gris_32(reducido), miniatura, ancho, alto


class ArbolBK:
    def __init__(self):
        self.raiz = None

    def agregar(self, valor):
        if self.raiz is None:
            self.raiz = (valor, {})
            return
        actual = self.raiz
        while True:
            v, hijos = actual
            d = (v ^ valor).bit_count()
            if d == 0:
                return
            if d not in hijos:
                hijos[d] = (valor, {})
                return
            actual = hijos[d]

    def buscar(self, valor, distancia_maxima):
        if self.raiz is None:
            return []
        res, pendientes = [], [self.raiz]
        while pendientes:
            v, hijos = pendientes.pop()
            distancia = (v ^ valor).bit_count()
            if distancia <= distancia_maxima:
                res.append((v, distancia))
            pendientes.extend(nodo for d, nodo in hijos.items()
                              if distancia-distancia_maxima <= d <= distancia+distancia_maxima)
        return res


# Las imágenes conservadas se consultan en el conjunto original.
raiz_lectura = ORIGEN
residuales = conservadas.sort_values("ruta_relativa_imagen").reset_index(drop=True).copy()
residuales["ruta_imagen_actual"] = residuales["ruta_relativa_imagen"].map(
    lambda r: str(raiz_lectura / Path(r))
)
residuales["ruta_mascara_actual"] = residuales["ruta_relativa_mascara"].map(
    lambda r: str(raiz_lectura / Path(r))
)

huellas, miniaturas, anchos, altos = [], [], [], []
for numero, fila in enumerate(residuales.itertuples(index=False), 1):
    huella, miniatura, ancho, alto = miniaturas_y_huella(Path(fila.ruta_imagen_actual))
    huellas.append(huella); miniaturas.append(miniatura)
    anchos.append(ancho); altos.append(alto)
    if numero % 500 == 0 or numero == len(residuales):
        print(f"Lectura perceptual: {numero:,}/{len(residuales):,}")

arbol = ArbolBK()
indice_huellas = defaultdict(list)
candidatos = []
for i, huella in enumerate(huellas):
    for huella_previa, distancia in arbol.buscar(huella, DISTANCIA_PHASH_MAX):
        for j in indice_huellas[huella_previa]:
            ri = anchos[i]/altos[i]
            rj = anchos[j]/altos[j]
            if abs(ri-rj) > DIFERENCIA_MAX_ASPECTO:
                continue
            similitud = float(structural_similarity(miniaturas[j], miniaturas[i], data_range=255))
            if similitud >= SSIM_MINIMO:
                uno, dos = residuales.iloc[j], residuales.iloc[i]
                candidatos.append({
                    "ruta_relativa_1": uno.ruta_relativa_imagen,
                    "ruta_relativa_2": dos.ruta_relativa_imagen,
                    "particion_1": uno.particion, "particion_2": dos.particion,
                    "tipo_tumor_1": uno.tipo_tumor, "tipo_tumor_2": dos.tipo_tumor,
                    "plano_1": uno.plano, "plano_2": dos.plano,
                    "ruta_imagen_1": uno.ruta_imagen_actual, "ruta_imagen_2": dos.ruta_imagen_actual,
                    "ruta_mascara_1": uno.ruta_mascara_actual, "ruta_mascara_2": dos.ruta_mascara_actual,
                    "distancia_phash": int(distancia), "ssim": similitud,
                    "cruce_particiones": uno.particion != dos.particion,
                    "discordancia_etiquetas": ((uno.tipo_tumor, uno.plano) != (dos.tipo_tumor, dos.plano)),
                })
    if huella not in indice_huellas:
        arbol.agregar(huella)
    indice_huellas[huella].append(i)
    if (i+1) % 500 == 0 or i+1 == len(residuales):
        print(f"Búsqueda perceptual: {i+1:,}/{len(residuales):,}")

COLUMNAS_CANDIDATOS = [
    "ruta_relativa_1", "ruta_relativa_2", "particion_1", "particion_2",
    "tipo_tumor_1", "tipo_tumor_2", "plano_1", "plano_2",
    "ruta_imagen_1", "ruta_imagen_2", "ruta_mascara_1", "ruta_mascara_2",
    "distancia_phash", "ssim", "cruce_particiones", "discordancia_etiquetas",
]
pares_perceptuales_restantes = pd.DataFrame(candidatos, columns=COLUMNAS_CANDIDATOS)
if not pares_perceptuales_restantes.empty:
    pares_perceptuales_restantes = pares_perceptuales_restantes.sort_values(
        ["cruce_particiones", "ssim"], ascending=[False, False]
    ).reset_index(drop=True)

pares_perceptuales_restantes.to_csv(
    DIR_REGISTROS / "pares_perceptuales_restantes.csv", index=False, encoding="utf-8-sig"
)
print(f"\nCandidatos perceptuales que permanecen: {len(pares_perceptuales_restantes)}")
print(f"Cruces entre train y test: {int(pares_perceptuales_restantes.cruce_particiones.sum())}")
display(pares_perceptuales_restantes[[
    "ruta_relativa_1", "ruta_relativa_2", "distancia_phash", "ssim", "cruce_particiones"
]].head(30))


## 6. Selección de pares perceptuales para depuración

Se registran los candidatos perceptuales seleccionados para aplicar las reglas jerárquicas. La inspección visual debe preceder a la aplicación de las reglas de depuración sobre estos candidatos.


In [ ]:
if (N_PARES_PERCEPTUALES_ESPERADOS is not None and
        len(pares_perceptuales_restantes) != N_PARES_PERCEPTUALES_ESPERADOS):
    raise RuntimeError(
        f"Se encontraron {len(pares_perceptuales_restantes)} pares perceptuales, "
        f"pero se esperaban {N_PARES_PERCEPTUALES_ESPERADOS}. "
        "Verificar los parámetros y revisar galerías antes de depurar."
    )

# Se registran los candidatos seleccionados para depuración perceptual.
rutas_activas = dict(zip(
    conservadas.ruta_relativa_imagen.str.lower(), conservadas.id.astype(int)
))
COLUMNAS_CONFIRMADOS = [
    "ruta_relativa_1", "ruta_relativa_2", "id_1", "id_2",
    "particion_1", "particion_2", "distancia_phash_revision", "ssim_revision"
]
confirmados_rows = []
if APLICAR_PERCEPTUALES_CONFIRMADOS:
    for par in pares_perceptuales_restantes.itertuples(index=False):
        a, b = par.ruta_relativa_1.lower(), par.ruta_relativa_2.lower()
        if a not in rutas_activas or b not in rutas_activas:
            raise AssertionError("Par perceptual no localizado en inventario activo")
        confirmados_rows.append({
            "ruta_relativa_1": par.ruta_relativa_1,
            "ruta_relativa_2": par.ruta_relativa_2,
            "id_1": int(rutas_activas[a]), "id_2": int(rutas_activas[b]),
            "particion_1": par.particion_1,
            "particion_2": par.particion_2,
            "distancia_phash_revision": int(par.distancia_phash),
            "ssim_revision": float(par.ssim),
        })

pares_confirmados = pd.DataFrame(confirmados_rows, columns=COLUMNAS_CONFIRMADOS)
pares_confirmados.to_csv(
    DIR_REGISTROS / "pares_perceptuales_confirmados_aplicados.csv",
    index=False, encoding="utf-8-sig"
)
print("Candidatos perceptuales detectados:", len(pares_perceptuales_restantes))
print("Pares aprobados para aplicar R1–R3:", len(pares_confirmados))
display(pares_perceptuales_restantes[[
    "ruta_relativa_1", "ruta_relativa_2", "distancia_phash", "ssim",
    "cruce_particiones"
]].head(40))


## 7. Depuración de duplicados perceptuales

Los pares seleccionados se agrupan por conectividad y se evalúan conforme a R1, R2 y R3. Cuando existen resoluciones distintas, las máscaras se comparan sobre una cuadrícula común.


In [ ]:
# Agrupación transitiva de perceptuales confirmados, sin volver a incorporar excluidas exactas.
ids_activos = set(conservadas.id.astype(int))
padre_p = {i: i for i in ids_activos}

def raiz_p(i):
    while padre_p[i] != i:
        padre_p[i] = padre_p[padre_p[i]]
        i = padre_p[i]
    return i

def unir_p(i, j):
    a, b = raiz_p(i), raiz_p(j)
    if a != b:
        padre_p[max(a,b)] = min(a,b)

for par in pares_confirmados.itertuples(index=False):
    assert par.id_1 in ids_activos and par.id_2 in ids_activos
    unir_p(int(par.id_1), int(par.id_2))

comp_p = defaultdict(list)
for idx in ids_activos:
    comp_p[raiz_p(idx)].append(idx)
grupos_perceptuales = sorted((sorted(v) for v in comp_p.values() if len(v) > 1),
                            key=lambda v: inventario.at[min(v), "ruta_relativa_imagen"])

# En imágenes con resoluciones distintas, el área en píxeles original NO es comparable.
# Se compara el área en una cuadrícula común, sin interpolación bilineal de etiquetas.
mascaras_en_cuadricula = {}

def mascara_normalizada(idx):
    if idx not in mascaras_en_cuadricula:
        a = cargar_mascara_binaria(idx)
        with Image.fromarray(a.astype(np.uint8)*255) as img:
            resized = img.resize(TAM_DICE_PERCEPTUAL, resample=Image.Resampling.NEAREST)
            mascaras_en_cuadricula[idx] = np.asarray(resized) > 0
    return mascaras_en_cuadricula[idx]


def dice_grupo_perceptual(i, j):
    a, b = cargar_mascara_binaria(i), cargar_mascara_binaria(j)
    if a.shape == b.shape:
        return dice_binario(a, b), "dimensiones_iguales"
    # Solo válido si las imágenes representan el mismo campo anatómico en ambos tamaños.
    return dice_binario(mascara_normalizada(i), mascara_normalizada(j)), "redimensionamiento_nn"


def clave_prioridad_perceptual(idx, resoluciones_diferentes):
    if resoluciones_diferentes:
        area = int(mascara_normalizada(idx).sum())
    else:
        area = int(inventario.at[idx, "area_tumoral_pixeles"])
    return (
        particion_prioridad[inventario.at[idx, "particion"]],
        -area,
        inventario.at[idx, "ruta_relativa_imagen"].lower()
    )

# Copiar el resultado de la etapa exacta (no cambiar sus reglas ni motivos).
decision_final = {int(k): dict(v) for k, v in decision.items()}
resumen_perceptuales, comparaciones_perceptuales = [], []

for num, ids in enumerate(grupos_perceptuales, 1):
    grupo = f"P{num:05d}"
    filas = inventario.loc[ids]
    conflicto = filas[["tipo_tumor", "plano"]].drop_duplicates().shape[0] > 1
    formas = {(int(inventario.at[i, "alto"]), int(inventario.at[i, "ancho"])) for i in ids}
    cambio_resolucion = len(formas) > 1
    dice_min, peor_par = np.nan, (None, None)
    metodos_dice = set()

    if conflicto:
        regla = "R1_perceptual_etiquetas_inconsistentes"
        elegido = None
    else:
        for i, j in combinations(ids, 2):
            d, metodo = dice_grupo_perceptual(i, j)
            metodos_dice.add(metodo)
            comparaciones_perceptuales.append({
                "grupo_perceptual": grupo,
                "id_1": i, "id_2": j,
                "ruta_1": inventario.at[i, "ruta_relativa_imagen"],
                "ruta_2": inventario.at[j, "ruta_relativa_imagen"],
                "dice": float(d), "metodo_alineacion": metodo,
            })
            if np.isnan(dice_min) or d < dice_min:
                dice_min, peor_par = float(d), (i,j)
        if dice_min < UMBRAL_DICE_MINIMO:
            regla = "R2_perceptual_dice_inferior_umbral"
            elegido = None
        else:
            regla = "R3_perceptual_copias_aceptadas"
            elegido = min(ids, key=lambda i: clave_prioridad_perceptual(i, cambio_resolucion))

    for i in ids:
        if elegido is None:
            decision_final[i].update(decision="excluir", motivo=regla, grupo_perceptual=grupo)
        elif i == elegido:
            decision_final[i].update(decision="conservar", motivo="R3_perceptual_representativa", grupo_perceptual=grupo)
        else:
            decision_final[i].update(decision="excluir", motivo="R3_perceptual_redundante", grupo_perceptual=grupo)

    resumen_perceptuales.append({
        "grupo_perceptual": grupo, "n_miembros": len(ids),
        "discordancia_etiquetas": bool(conflicto),
        "resoluciones_diferentes": bool(cambio_resolucion),
        "dice_minimo": dice_min,
        "umbral_dice": UMBRAL_DICE_MINIMO,
        "peor_par_id_1": peor_par[0], "peor_par_id_2": peor_par[1],
        "metodos_dice": ";".join(sorted(metodos_dice)),
        "regla_aplicada": regla,
        "ruta_conservada": inventario.at[elegido,"ruta_relativa_imagen"] if elegido is not None else "",
        "n_eliminadas": len(ids) if elegido is None else len(ids)-1,
    })

COLUMNAS_GRUPOS_P = [
    "grupo_perceptual", "n_miembros", "discordancia_etiquetas",
    "resoluciones_diferentes", "dice_minimo", "umbral_dice",
    "peor_par_id_1", "peor_par_id_2", "metodos_dice", "regla_aplicada",
    "ruta_conservada", "n_eliminadas",
]
COLUMNAS_DICE_P = ["grupo_perceptual", "id_1", "id_2", "ruta_1", "ruta_2", "dice", "metodo_alineacion"]
resumen_grupos_perceptuales = pd.DataFrame(resumen_perceptuales, columns=COLUMNAS_GRUPOS_P)
pares_dice_perceptuales = pd.DataFrame(comparaciones_perceptuales, columns=COLUMNAS_DICE_P)
resumen_grupos_perceptuales.to_csv(DIR_REGISTROS / "grupos_perceptuales_y_reglas.csv", index=False, encoding="utf-8-sig")
pares_dice_perceptuales.to_csv(DIR_REGISTROS / "comparaciones_dice_perceptuales.csv", index=False, encoding="utf-8-sig")

registro_final = inventario.copy()
registro_final["grupo_exacto"] = [decision_final[i].get("grupo_exacto", "") for i in range(n)]
registro_final["grupo_perceptual"] = [decision_final[i].get("grupo_perceptual", "") for i in range(n)]
registro_final["tipo_duplicado"] = np.select(
    [registro_final["grupo_perceptual"].ne(""), registro_final["grupo_exacto"].ne("")],
    ["perceptual", "exacto"], default="sin_duplicado"
)
registro_final["decision"] = [decision_final[i]["decision"] for i in range(n)]
registro_final["motivo"] = [decision_final[i]["motivo"] for i in range(n)]
registro_final["regla_aplicada"] = registro_final["motivo"].str.extract(r"^(R[123])", expand=False).fillna("")
rutas_representantes = {}
for f in resumen_grupos:
    rutas_representantes[f["grupo_exacto"]] = f["ruta_conservada"]
for f in resumen_perceptuales:
    rutas_representantes[f["grupo_perceptual"]] = f["ruta_conservada"]
registro_final["imagen_conservada_del_grupo"] = [
    rutas_representantes.get(gp, "") if gp else rutas_representantes.get(ge, "")
    for ge, gp in zip(registro_final.grupo_exacto, registro_final.grupo_perceptual)
]
registro_final["ruta_nueva_imagen"] = registro_final.ruta_relativa_imagen.map(lambda r: str(DESTINO / Path(r)))
registro_final["ruta_nueva_mascara"] = registro_final.ruta_relativa_mascara.map(lambda r: str(DESTINO / Path(r)))
conservadas = registro_final.loc[registro_final.decision == "conservar"].copy()
excluidas = registro_final.loc[registro_final.decision == "excluir"].copy()

for nombre, data in [
    ("decisiones_por_imagen.csv", registro_final),
    ("imagenes_conservadas.csv", conservadas),
    ("imagenes_excluidas.csv", excluidas),
    ("resumen_decisiones.csv", registro_final.groupby(["decision", "motivo"]).size().rename("n_imagenes").reset_index()),
]:
    data.to_csv(DIR_REGISTROS / nombre, index=False, encoding="utf-8-sig")

assert len(conservadas) + len(excluidas) == len(inventario)
assert set(conservadas.id).isdisjoint(set(excluidas.id))
assert not conservadas.duplicated("sha256_pixeles").any()
assert not conservadas.duplicated("sha256_archivo").any()
for ids in grupos_perceptuales:
    assert (registro_final.loc[ids, "decision"] == "conservar").sum() <= 1
for par in pares_confirmados.itertuples(index=False):
    assert not (par.id_1 in set(conservadas.id) and par.id_2 in set(conservadas.id))

print(f"Grupos perceptuales confirmados: {len(grupos_perceptuales)}")
print(f"Imágenes conservadas etapa exacta: {len(ids_activos)}")
print(f"Excluidas en etapa perceptual: {len(ids_activos)-len(conservadas)}")
print(f"Total excluidas: {len(excluidas)}, total conservadas: {len(conservadas)}")
display(registro_final.groupby(["decision", "motivo"]).size().rename("n").reset_index())
display(resumen_grupos_perceptuales)


## 8. Galerías de duplicados perceptuales

Se guardan las figuras de los grupos perceptuales, incluyendo las decisiones de conservación o exclusión y las comparaciones de máscaras pertinentes.


In [ ]:
preparar_directorios_galerias(DIR_PERCEPTUALES)
indices_por_grupo_perceptual = {
    f"P{k:05d}": ids for k, ids in enumerate(grupos_perceptuales, 1)
}

# Recuperar los valores de pHash y SSIM calculados en la búsqueda perceptual.
# Solo se presentan las parejas realmente medidas y confirmadas.
metricas_por_par = {
    frozenset((int(par.id_1), int(par.id_2))): (
        float(par.ssim_revision), int(par.distancia_phash_revision)
    )
    for par in pares_confirmados.itertuples(index=False)
}

ejemplos_perceptuales = {r: [] for r in SUBCARPETAS_REGLAS}
for fila in resumen_grupos_perceptuales.itertuples(index=False):
    regla = fila.regla_aplicada[:2]
    ids = indices_por_grupo_perceptual[fila.grupo_perceptual]
    elegido = next((i for i in ids if decision_final[i]["decision"] == "conservar"), None)
    metricas_grupo = [
        (a, b, *metricas_por_par[frozenset((a, b))])
        for a, b in combinations(sorted(ids), 2)
        if frozenset((a, b)) in metricas_por_par
    ]
    figura = crear_galeria_grupo(
        ids, "perceptual", fila.grupo_perceptual, regla,
        elegido=elegido, dice_minimo=fila.dice_minimo,
        peor_par=(fila.peor_par_id_1, fila.peor_par_id_2),
        diferentes_dimensiones=bool(fila.resoluciones_diferentes),
        metricas_perceptuales=metricas_grupo,
    )
    ejemplos_perceptuales[regla].append(figura)
mostrar_ejemplo_por_regla(ejemplos_perceptuales, "Galerías: duplicados perceptuales")


## 9. Balance antes y después de la depuración

Se calculan frecuencias, proporciones y reducciones por partición, tipo de tumor y plano anatómico. Los resultados se exportan como tablas y gráficos.


In [ ]:
ETIQUETAS_TUMOR = ["glioma", "meningioma", "pituitario"]
ETIQUETAS_PLANO = ["axial", "coronal", "sagital"]


def tabla_balance(columnas, nombre, por_particion=False):
    cols = (["particion"] if por_particion else []) + list(columnas)
    niveles = []
    for campo in cols:
        if campo == "particion": niveles.append(PARTICIONES)
        elif campo == "tipo_tumor": niveles.append(ETIQUETAS_TUMOR)
        elif campo == "plano": niveles.append(ETIQUETAS_PLANO)
    # Para una sola variable se requiere un Index sencillo; un MultiIndex
    # de un nivel no coincide con el índice de groupby y produce ceros falsos.
    if len(cols) == 1:
        todas = pd.Index(niveles[0], name=cols[0])
    else:
        todas = pd.MultiIndex.from_product(niveles, names=cols)
    a = inventario.groupby(cols).size().reindex(todas, fill_value=0).rename("antes")
    d = conservadas.groupby(cols).size().reindex(todas, fill_value=0).rename("despues")
    tabla = pd.concat([a, d], axis=1).reset_index()
    tabla["eliminadas"] = tabla["antes"] - tabla["despues"]
    tabla["reduccion_pct"] = np.where(tabla["antes"] > 0,
                                       100*tabla["eliminadas"]/tabla["antes"], np.nan)
    if por_particion and columnas:
        # Balance dentro de cada partición para tumor, plano o tumor–plano.
        nantes = tabla.groupby("particion")["antes"].transform("sum")
        ndespues = tabla.groupby("particion")["despues"].transform("sum")
    else:
        # Al comparar train frente a test, los porcentajes deben sumar 100 %
        # sobre todo el dataset, no ser 100 % para cada partición.
        nantes, ndespues = len(inventario), len(conservadas)
    tabla["participacion_antes_pct"] = np.where(nantes > 0, 100*tabla["antes"]/nantes, np.nan)
    tabla["participacion_despues_pct"] = np.where(ndespues > 0, 100*tabla["despues"]/ndespues, np.nan)
    tabla["cambio_participacion_pp"] = tabla["participacion_despues_pct"] - tabla["participacion_antes_pct"]
    tabla.to_csv(DIR_BALANCE / f"balance_{nombre}.csv", index=False, encoding="utf-8-sig")
    return tabla

balances = {
    "tipo_tumor": tabla_balance(["tipo_tumor"], "tipo_tumor"),
    "plano": tabla_balance(["plano"], "plano"),
    "tumor_plano": tabla_balance(["tipo_tumor", "plano"], "tumor_plano"),
    "particion": tabla_balance([], "particion", por_particion=True),
    "particion_tumor": tabla_balance(["tipo_tumor"], "particion_tumor", por_particion=True),
    "particion_plano": tabla_balance(["plano"], "particion_plano", por_particion=True),
    "particion_tumor_plano": tabla_balance(["tipo_tumor", "plano"], "particion_tumor_plano", por_particion=True),
}
for titulo in ("particion", "tipo_tumor", "plano", "tumor_plano", "particion_tumor_plano"):
    print("\nBALANCE:", titulo)
    display(balances[titulo].round(2))

# -------------------------------------------------------------
# Gráficos de balance: cantidades + composición porcentual.
# -------------------------------------------------------------

def etiqueta_balance(fila, columnas):
    """Categorías legibles y en orden fijo para los gráficos."""
    etiquetas = []
    for columna in columnas:
        if columna == "particion":
            etiquetas.append(str(fila[columna]).capitalize())
        else:
            etiquetas.append(str(fila[columna]).capitalize())
    return "\n".join(etiquetas)


def graficar_balance(tabla, columnas, titulo, archivo, separar_por_particion=False):
    """Compara valores absolutos y % antes/después, global o por set."""
    if separar_por_particion:
        subconjuntos = [(p, tabla.loc[tabla.particion == p].copy()) for p in PARTICIONES]
    else:
        subconjuntos = [(None, tabla.copy())]

    cantidad_filas = len(subconjuntos)
    max_categorias = max(len(t) for _, t in subconjuntos)
    ancho_figura = 16 if max_categorias >= 9 else (13 if max_categorias >= 6 else 11)
    fig, ejes = plt.subplots(
        cantidad_filas, 2,
        figsize=(ancho_figura, 4.8 * cantidad_filas),
        squeeze=False,
    )

    for fila_idx, (particion, datos) in enumerate(subconjuntos):
        categorias = [etiqueta_balance(r, columnas) for _, r in datos.iterrows()]
        x = np.arange(len(datos))
        ancho = 0.39
        previo, posterior = datos["antes"].to_numpy(), datos["despues"].to_numpy()
        porcentajes_previos = datos["participacion_antes_pct"].fillna(0).to_numpy()
        porcentajes_posteriores = datos["participacion_despues_pct"].fillna(0).to_numpy()
        subtitulo = f" — {particion.capitalize()}" if particion else ""

        # Panel izquierdo: números absolutos.
        ax = ejes[fila_idx, 0]
        barras_antes = ax.bar(x - ancho / 2, previo, width=ancho, label="Original")
        barras_despues = ax.bar(x + ancho / 2, posterior, width=ancho, label="Depurado")
        ax.set_title(f"Número de imágenes{subtitulo}")
        ax.set_ylabel("Número de imágenes")
        ax.set_xticks(x, categorias)
        ax.tick_params(axis="x", labelsize=8)
        ax.set_ylim(0, max(1, int(max(np.max(previo), np.max(posterior)) * 1.20)))
        ax.legend(loc="upper right")
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        for barras in (barras_antes, barras_despues):
            ax.bar_label(barras, labels=[f"{v:.0f}" if v > 0 else "" for v in barras.datavalues], padding=2, fontsize=7)

        # Panel derecho: proporciones (dentro de la partición si procede).
        ax = ejes[fila_idx, 1]
        b_antes = ax.bar(x - ancho / 2, porcentajes_previos, width=ancho, label="Original")
        b_despues = ax.bar(x + ancho / 2, porcentajes_posteriores, width=ancho, label="Depurado")
        referencia = "en el conjunto" if not separar_por_particion else "dentro de la partición"
        ax.set_title(f"Participación porcentual {referencia}{subtitulo}")
        ax.set_ylabel("Participación (%)")
        ax.set_xticks(x, categorias)
        ax.tick_params(axis="x", labelsize=8)
        ax.set_ylim(0, max(10, float(max(np.max(porcentajes_previos), np.max(porcentajes_posteriores))) * 1.25))
        ax.legend(loc="upper right")
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        for barras in (b_antes, b_despues):
            ax.bar_label(barras, labels=[f"{v:.1f}" if v > 0 else "" for v in barras.datavalues], padding=2, fontsize=7)

    fig.suptitle(titulo, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    destino = DIR_BALANCE / archivo
    fig.savefig(destino, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return destino


especificaciones_graficos = [
    ("tipo_tumor", ["tipo_tumor"], "Balance por tipo de tumor", "grafico_balance_tipo_tumor.png", False),
    ("plano", ["plano"], "Balance por plano anatómico", "grafico_balance_plano.png", False),
    ("tumor_plano", ["tipo_tumor", "plano"], "Balance de las nueve condiciones tumor–plano", "balance_9_condiciones.png", False),
    ("particion", ["particion"], "Balance entre particiones", "grafico_balance_particiones.png", False),
    ("particion_tumor", ["tipo_tumor"], "Balance por tumor dentro de cada partición", "grafico_balance_particion_tumor.png", True),
    ("particion_plano", ["plano"], "Balance por plano dentro de cada partición", "grafico_balance_particion_plano.png", True),
    ("particion_tumor_plano", ["tipo_tumor", "plano"], "Balance por tumor–plano dentro de cada partición", "grafico_balance_particion_tumor_plano.png", True),
]

archivos_graficos = []
for clave, columnas, titulo, archivo, por_particion in especificaciones_graficos:
    archivos_graficos.append(
        graficar_balance(
            balances[clave], columnas, titulo, archivo,
            separar_por_particion=por_particion,
        )
    )

# Gráfico complementario: porcentaje eliminado dentro de cada condición.
fig, ax = plt.subplots(figsize=(13, 5))
datos = balances["tumor_plano"]
labels = [etiqueta_balance(fila, ["tipo_tumor", "plano"]) for _, fila in datos.iterrows()]
columnas = ax.bar(np.arange(len(datos)), datos["reduccion_pct"].fillna(0))
ax.set_xticks(np.arange(len(datos)), labels)
ax.set_ylabel("Imágenes eliminadas respecto del total original (%)")
ax.set_title("Reducción porcentual por combinación tumor–plano")
ax.set_ylim(0, max(2, float(datos["reduccion_pct"].fillna(0).max()) * 1.2))
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.bar_label(columnas, labels=[f"{v:.1f}%" if pd.notna(v) and v > 0 else "" for v in datos["reduccion_pct"]], padding=3, fontsize=8)
fig.tight_layout()
archivo_reduccion = DIR_BALANCE / "grafico_reduccion_por_tumor_plano.png"
fig.savefig(archivo_reduccion, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
archivos_graficos.append(archivo_reduccion)

print(f"Se guardaron {len(archivos_graficos)} gráficos de balance en: {DIR_BALANCE}")
for archivo in archivos_graficos:
    print(" -", archivo.name)


## 10. Creación opcional del conjunto depurado

La copia de los pares conservados se realiza únicamente si se habilita la exportación y no existe un conjunto depurado previo. Los datos originales permanecen inalterados.


In [ ]:
TEMPORAL_ROOT = DESTINO_ROOT.with_name(DESTINO_ROOT.name + "__en_construccion")

def crear_copia_depurada():
    if DESTINO_ROOT.exists():
        raise FileExistsError(
            f"Ya existe la carpeta destino: {DESTINO_ROOT}. No se sobrescribirá. "
            "Para reconstruir, mover o renombrar manualmente la versión anterior."
        )
    if TEMPORAL_ROOT.exists():
        raise FileExistsError(
            f"Hay una carpeta temporal de una ejecución anterior: {TEMPORAL_ROOT}. "
            "Revisarla y resolverla manualmente antes de volver a ejecutar."
        )
    TEMPORAL_ROOT.parent.mkdir(parents=True, exist_ok=True)
    TEMPORAL_ROOT.mkdir(parents=True, exist_ok=False)
    temporal_seg = TEMPORAL_ROOT / "segmentation_task"
    for particion in PARTICIONES:
        (temporal_seg / particion / "images").mkdir(parents=True, exist_ok=True)
        (temporal_seg / particion / "masks").mkdir(parents=True, exist_ok=True)

    try:
        for numero, fila in enumerate(conservadas.itertuples(index=False), 1):
            for origen_txt, relativa in (
                (fila.ruta_imagen_original, fila.ruta_relativa_imagen),
                (fila.ruta_mascara_original, fila.ruta_relativa_mascara),
            ):
                fuente = Path(origen_txt)
                destino = temporal_seg / Path(relativa)
                destino.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(fuente, destino)
                if fuente.stat().st_size != destino.stat().st_size:
                    raise IOError(f"Tamaño diferente al copiar {fuente}")
                if sha_archivo(fuente) != sha_archivo(destino):
                    raise IOError(f"SHA-256 diferente al copiar {fuente}")
            if numero % 500 == 0 or numero == len(conservadas):
                print(f"Pares copiados/verificados: {numero:,}/{len(conservadas):,}")

        nimg = sum(1 for p in temporal_seg.rglob("*") if p.is_file() and p.parent.name == "images")
        nmask = sum(1 for p in temporal_seg.rglob("*") if p.is_file() and p.parent.name == "masks")
        if (nimg, nmask) != (len(conservadas), len(conservadas)):
            raise AssertionError(f"Conteos en staging: imágenes={nimg}, máscaras={nmask}")
        if DESTINO_ROOT.exists():
            raise FileExistsError(f"Se creó el destino durante la copia: {DESTINO_ROOT}")
        TEMPORAL_ROOT.rename(DESTINO_ROOT)
        print("Dataset depurado creado y verificado:", DESTINO)
    except Exception:
        print("No se modificó el dataset original.")
        print("La copia incompleta, si existe, permanece en:", TEMPORAL_ROOT)
        raise


if CREAR_DATASET_DEPURADO:
    crear_copia_depurada()
else:
    print("Modo sin copia: decisiones y balances calculados, sin crear BRISC2025")


## 11. Registro de las decisiones y trazabilidad

Se exportan el inventario, las decisiones de depuración, las comparaciones de máscaras y la configuración del procedimiento.


In [ ]:
# Registros resumidos por tipo y regla (no confundir pares con imágenes eliminadas).
filas_resumen = []
for tipo, tabla, campo in (
    ("exacto", grupos, "grupo_exacto"),
    ("perceptual", resumen_grupos_perceptuales, "grupo_perceptual"),
):
    for fila in tabla.itertuples(index=False):
        regla = fila.regla_aplicada[:2]
        filas_resumen.append({
            "tipo_duplicado": tipo,
            "regla_aplicada": regla,
            "grupo": getattr(fila, campo),
            "n_miembros": int(fila.n_miembros),
            "n_eliminadas": int(fila.n_eliminadas),
        })
detalle_grupos = pd.DataFrame(
    filas_resumen,
    columns=["tipo_duplicado", "regla_aplicada", "grupo", "n_miembros", "n_eliminadas"]
)
resumen_por_regla = (
    detalle_grupos.groupby(["tipo_duplicado", "regla_aplicada"])
    .agg(n_grupos=("grupo", "size"), n_imagenes_eliminadas=("n_eliminadas", "sum"))
    .reset_index()
)
for tipo in ("exacto", "perceptual"):
    for regla in ("R1", "R2", "R3"):
        if not ((resumen_por_regla.tipo_duplicado == tipo) &
                (resumen_por_regla.regla_aplicada == regla)).any():
            resumen_por_regla.loc[len(resumen_por_regla)] = [tipo, regla, 0, 0]
resumen_por_regla = resumen_por_regla.sort_values(
    ["tipo_duplicado", "regla_aplicada"]
).reset_index(drop=True)
resumen_por_regla.to_csv(DIR_REGISTROS / "resumen_por_regla.csv", index=False, encoding="utf-8-sig")
detalle_grupos.to_csv(DIR_REGISTROS / "detalle_grupos.csv", index=False, encoding="utf-8-sig")

resumen_general = pd.DataFrame([{
    "imagenes_originales": len(inventario),
    "excluidas_exactos": len(inventario) - len(ids_activos),
    "excluidas_perceptuales": len(ids_activos) - len(conservadas),
    "excluidas_total": len(excluidas),
    "conservadas_total": len(conservadas),
    "grupos_exactos": len(componentes_duplicados),
    "pares_perceptuales_detectados": len(pares_perceptuales_restantes),
    "pares_perceptuales_confirmados": len(pares_confirmados),
    "grupos_perceptuales_confirmados": len(grupos_perceptuales),
}])
resumen_general.to_csv(DIR_REGISTROS / "resumen_general.csv", index=False, encoding="utf-8-sig")

configuracion = {
    "origen": "data/BRISC2025_original/segmentation_task",
    "destino": "data/BRISC2025/segmentation_task",
    "salida": "results/OE0/depuracion_duplicados",
    "umbral_mascara": UMBRAL_MASCARA,
    "umbral_dice_minimo": UMBRAL_DICE_MINIMO,
    "distancia_phash_max": DISTANCIA_PHASH_MAX,
    "ssim_minimo": SSIM_MINIMO,
    "diferencia_max_aspecto": DIFERENCIA_MAX_ASPECTO,
    "tam_ssim": TAM_SSIM,
    "tam_dice_perceptual": TAM_DICE_PERCEPTUAL,
    "aplicar_perceptuales_confirmados": APLICAR_PERCEPTUALES_CONFIRMADOS,
    "pares_perceptuales_esperados": N_PARES_PERCEPTUALES_ESPERADOS,
    "crear_dataset_depurado": CREAR_DATASET_DEPURADO,
    "dataset_depurado_creado": DESTINO_ROOT.is_dir(),
    "seleccion": "test > train; dentro de la particion: area tumoral mayor; empate: ruta alfabetica",
    "reglas": {
        "R1": "etiquetas tumor o plano inconsistentes: excluir grupo",
        "R2": "Dice minimo < 0.80: excluir grupo",
        "R3": "conservar una copia y excluir redundantes",
    },
}
(SALIDA / "configuracion.json").write_text(
    json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
)

informe = f"""# Depuración de duplicados de BRISC

- Original: `data/BRISC2025_original/segmentation_task`.
- Dataset depurado: `data/BRISC2025/segmentation_task`.
- Resultados: `results/OE0/depuracion_duplicados`.
- Imágenes originales: **{len(inventario)}**.
- Grupos exactos: **{len(componentes_duplicados)}**.
- Pares perceptuales detectados: **{len(pares_perceptuales_restantes)}**.
- Pares perceptuales confirmados en esta ejecución: **{len(pares_confirmados)}**.
- Excluidas por exactos: **{len(inventario)-len(ids_activos)}**.
- Excluidas por perceptuales: **{len(ids_activos)-len(conservadas)}**.
- **Total excluidas: {len(excluidas)}; conservadas: {len(conservadas)}.**

## Reglas jerárquicas

1. Si difiere tumor o plano, excluir todo el grupo (R1).
2. Si las etiquetas coinciden pero el mínimo Dice entre máscaras binarias (`> {UMBRAL_MASCARA}`) es inferior a {UMBRAL_DICE_MINIMO:.2f}, excluir el grupo (R2).
3. Si el grupo supera ambas reglas, conservar una imagen priorizando test; dentro de un mismo set, mayor área tumoral; ante empate, ruta alfabética (R3).

Los pares perceptuales se proponen con pHash ≤ {DISTANCIA_PHASH_MAX}, SSIM ≥ {SSIM_MINIMO:.2f} y diferencia absoluta máxima de relación de aspecto de {DIFERENCIA_MAX_ASPECTO:.2f}. Su condición de duplicados **requiere confirmación visual**, especialmente cuando se cambian los parámetros. Cuando las resoluciones difieren, las máscaras se redimensionan con vecino más cercano sin registro espacial, por lo que se debe revisar la validez de la alineación.

La depuración garantiza eliminación de duplicados detectados según estos métodos, **no independencia por paciente**.

## Organización de resultados

- `01_duplicados_exactos/`: galerías de R1, R2 y R3.
- `02_duplicados_perceptuales/`: galerías de R1, R2 y R3.
- `03_tablas_balance/`: tablas CSV y gráficos PNG de frecuencias, proporciones y reducción porcentual antes/después.
- `04_registros_depuracion/`: inventarios, exclusiones, grupos, Dice y este informe.
- `configuracion.json`: configuración aplicada.
"""
(DIR_REGISTROS / "informe_depuracion.md").write_text(informe, encoding="utf-8")
print(informe)
display(resumen_por_regla)
display(resumen_general)
